# Gridlock Hackathon 2.0 — Traffic Demand Prediction

Traffic demand prediction for Day 49 test hours (2:15–13:45). The core idea is that Day 48 is a near-perfect structural proxy for Day 49. We use it as a baseline, then correct it by how much Day 49 has been drifting from Day 48 in the three hours we can observe (0, 1, 2).

Key components: per-geohash ratio extrapolation, ±15-min slot smoothing on Day 48 lookups, Day 49 slot-level anchors (2:00 and 1:45), Day 48 daily slope per geohash, geo4 spatial lag, a 5-seed × 5-fold LightGBM ensemble, and a light blend with the rule-based estimate.

In [1]:
!pip install lightgbm pygeohash scikit-learn pandas numpy -q

In [2]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
print('Ready!')

Ready!


## Data

In [3]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

for df in [train, test]:
    df['hour']   = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)

d48 = train[train['day'] == 48].copy()
d49 = train[train['day'] == 49].copy()

for df in [d48, d49]:
    df['geo4'] = df['geohash'].str[:4]
    df['geo3'] = df['geohash'].str[:3]

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'd48: {d48.shape} | d49: {d49.shape}')
print(f'd49 hours: {sorted(d49["hour"].unique())}  |  Test hours: {sorted(test["hour"].unique())}')
print(f'Global demand mean: {train["demand"].mean():.4f}')

## Lookup tables

Everything downstream depends on three things: an exact (geo, timestamp) lookup into d48, the per-geohash d49/d48 ratio fit across hours 0–2, and a handful of slot-level helpers added in this version.

In [4]:
# Exact timestamp and hour-level d48 lookups
d48_ts_dict = d48.groupby(['geohash', 'timestamp'])['demand'].mean().to_dict()
d48_hr_dict = d48.groupby(['geohash', 'hour'])['demand'].mean().to_dict()
d49_hr_dict = d49.groupby(['geohash', 'hour'])['demand'].mean().to_dict()

# Chronological ordering of all 96 d48 slots — needed for prev/next lookups
ts_sorted = sorted(
    d48['timestamp'].unique(),
    key=lambda x: int(x.split(':')[0]) * 60 + int(x.split(':')[1])
)
ts_idx = {ts: i for i, ts in enumerate(ts_sorted)}

# For each (geo, ts) build prev/next/roll3 dicts so we can smooth
# over point-in-time noise without per-row apply() at inference time
d48_prev_dict  = {}
d48_next_dict  = {}
d48_roll3_dict = {}

for i, ts in enumerate(ts_sorted):
    ts_prev = ts_sorted[max(0, i - 1)]
    ts_next = ts_sorted[min(len(ts_sorted) - 1, i + 1)]
    for geo in d48['geohash'].unique():
        v_c = d48_ts_dict.get((geo, ts))
        v_p = d48_ts_dict.get((geo, ts_prev))
        v_n = d48_ts_dict.get((geo, ts_next))
        if v_p is not None: d48_prev_dict[(geo, ts)]  = v_p
        if v_n is not None: d48_next_dict[(geo, ts)]  = v_n
        vals = [x for x in [v_p, v_c, v_n] if x is not None]
        if vals:            d48_roll3_dict[(geo, ts)] = float(np.mean(vals))

print(f'd48_prev_dict:  {len(d48_prev_dict):,} entries')
print(f'd48_roll3_dict: {len(d48_roll3_dict):,} entries')

In [5]:
# Ratio trend: fit ratio(hour) = intercept + slope * hour using d49/d48 at hours 0, 1, 2.
# Extrapolating this to the test hours is the single most important signal.
r_by_h = {}
for h in [0, 1, 2]:
    d48h   = d48[d48['hour'] == h].groupby('geohash')['demand'].mean()
    d49h   = d49[d49['hour'] == h].groupby('geohash')['demand'].mean()
    common = d48h.index.intersection(d49h.index)
    r_by_h[h] = (d49h[common] / (d48h[common] + 1e-8)).clip(0.1, 10)

all_geo_3h = set(r_by_h[0].index) & set(r_by_h[1].index) & set(r_by_h[2].index)
ratio_intercept_d, ratio_slope_d = {}, {}
for g in all_geo_3h:
    slope, intercept = np.polyfit([0, 1, 2], [r_by_h[0][g], r_by_h[1][g], r_by_h[2][g]], 1)
    ratio_intercept_d[g] = intercept
    ratio_slope_d[g]     = slope

# Fallback tiers: geo-level early mean → geo4 level → global default 1.5
d48_early      = d48[d48['hour'].isin([0, 1, 2])].groupby('geohash')['demand'].mean()
d49_early      = d49.groupby('geohash')['demand'].mean()
c2             = d48_early.index.intersection(d49_early.index)
geo_ratio_dict = (d49_early[c2] / (d48_early[c2] + 1e-8)).clip(0.1, 10).to_dict()

geo4_ratio_dict = (
    d49.groupby('geo4')['demand'].mean() /
    (d48[d48['hour'].isin([0, 1, 2])]
       .assign(geo4=lambda df: df['geohash'].str[:4])
       .groupby('geo4')['demand'].mean() + 1e-8)
).clip(0.1, 10).to_dict()

def get_ratio_at_hour(geo, geo4, hour):
    if geo in ratio_intercept_d:
        return float(np.clip(ratio_intercept_d[geo] + ratio_slope_d[geo] * hour, 0.1, 10))
    v = geo_ratio_dict.get(geo)
    if v is not None: return v
    v = geo4_ratio_dict.get(geo4)
    if v is not None: return v
    return 1.5

print(f'Geohashes with full 3-hr ratio fit: {len(all_geo_3h)}')
print(f'Mean ratio h0={r_by_h[0].mean():.3f}  h1={r_by_h[1].mean():.3f}  h2={r_by_h[2].mean():.3f}')

In [6]:
# d49 early-morning signals
def safe_slope(x):
    if x['hour'].nunique() < 2: return 0.0
    return float(np.polyfit(x['hour'], x['demand'], 1)[0])

d49_trend_dict = d49.groupby('geohash').apply(safe_slope).to_dict()
d49_last       = d49.sort_values('hour').groupby('geohash').last()[['demand', 'hour']]
d49_last_dict  = d49_last['demand'].to_dict()
d49_lhour_dict = d49_last['hour'].to_dict()

# The 2:0 slot is 15 min before the test window opens — it's the strongest anchor
d49_at_2_0_dict     = d49[d49['timestamp'] == '2:0'].groupby('geohash')['demand'].mean().to_dict()
d49_slot_last2_dict = (d49[d49['timestamp'].isin(['1:45', '2:0'])]
                          .groupby('geohash')['demand'].mean().to_dict())

# Geo × hour aggregates from d48 (historical context for each location)
geo_hour_d48 = d48.groupby(['geohash', 'hour'])['demand'].agg(
    geo_hour_mean='mean', geo_hour_std='std', geo_hour_med='median',
    geo_hour_q25=lambda x: x.quantile(0.25),
    geo_hour_q75=lambda x: x.quantile(0.75)
).reset_index()
geo_hour_d48['geo_hour_std'] = geo_hour_d48['geo_hour_std'].fillna(0)

d48['geo4'] = d48['geohash'].str[:4]
d48['geo3'] = d48['geohash'].str[:3]
geo4_hour   = (d48.groupby(['geo4', 'hour'])['demand'].mean()
                  .reset_index().rename(columns={'demand': 'geo4_hour_mean'}))
geo3_hour   = (d48.groupby(['geo3', 'hour'])['demand'].mean()
                  .reset_index().rename(columns={'demand': 'geo3_hour_mean'}))
geo_d48     = d48.groupby('geohash')['demand'].agg(
    geo_d48_mean='mean', geo_d48_std='std', geo_d48_med='median').reset_index()

d49_trend_df = pd.DataFrame({'geohash': list(d49_trend_dict),
                              'd49_trend': list(d49_trend_dict.values())})
d49_last_df  = pd.DataFrame({
    'geohash':         list(d49_last_dict),
    'd49_last_demand': list(d49_last_dict.values()),
    'd49_last_hour':   list(d49_lhour_dict.values())
})

r2_dict = r_by_h[2].to_dict()

# d48 daily slope: does this geo's demand rise or fall through the day?
d48_geo_hour_tmp   = d48.groupby(['geohash', 'hour'])['demand'].mean().reset_index()
def _slope(grp):
    if len(grp) < 3: return 0.0
    return float(np.polyfit(grp['hour'], grp['demand'], 1)[0])
d48_day_slope_dict = d48_geo_hour_tmp.groupby('geohash').apply(_slope).to_dict()

# Geo4 neighbor mean: mean d48 demand of sibling geohashes, excluding self
geo_own_d48_mean = d48.groupby('geohash')['demand'].mean().to_dict()
geo4_d48_mean    = d48.groupby('geo4')['demand'].mean().to_dict()
geo4_counts_d48  = d48.groupby('geo4')['geohash'].nunique().to_dict()

def get_geo4_neighbor_mean(geo, geo4):
    g4_mean   = geo4_d48_mean.get(geo4, np.nan)
    self_mean = geo_own_d48_mean.get(geo, g4_mean)
    n = geo4_counts_d48.get(geo4, 1)
    if n <= 1: return g4_mean if not np.isnan(g4_mean) else 0.09
    return (g4_mean * n - self_mean) / (n - 1)

# Lat/lon from geohash
all_geohashes = set(train['geohash']) | set(test['geohash'])
geo_coords    = {g: pgh.decode(g) for g in all_geohashes}
lat_dict      = {g: v.latitude  for g, v in geo_coords.items()}
lon_dict      = {g: v.longitude for g, v in geo_coords.items()}

print('Lookup tables ready')
print(f'd49 at 2:0 coverage: {len(d49_at_2_0_dict)} geos | d48 slope: {len(d48_day_slope_dict)} geos')

## Features

In [7]:
def make_features(df):
    df = df.copy()

    # Time
    df['hour']        = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']      = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_slot']   = df['hour'] * 4 + df['minute'] // 15
    df['time_of_day'] = df['hour'] + df['minute'] / 60.0
    df['hour_sin']    = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour']      / 24)
    df['slot_sin']    = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos']    = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['is_peak']     = ((df['hour'] >= 7)  & (df['hour'] <= 10)).astype(int)
    df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)

    # Location
    df['lat']  = df['geohash'].map(lat_dict)
    df['lon']  = df['geohash'].map(lon_dict)
    df['geo3'] = df['geohash'].str[:3]
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]

    # Road attributes
    df['LV_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['LM_enc'] = (df['Landmarks']     == 'Yes').astype(int)
    df['RT_enc'] = df['RoadType'].map({'Residential': 0, 'Street': 1, 'Highway': 2}).fillna(0)
    df['WX_enc'] = df['Weather'].map({'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3}).fillna(-1)
    df['Temperature'] = (df.groupby('geohash')['Temperature']
                           .transform(lambda x: x.fillna(x.median()))
                           .fillna(25.0))

    geo_ts_keys = list(zip(df['geohash'], df['timestamp']))
    geo_hr_keys = list(zip(df['geohash'], df['hour']))

    # d48 cross-day lookups
    df['d48_demand']  = pd.Series(
        [d48_ts_dict.get(k, d48_hr_dict.get((k[0], int(k[1].split(':')[0])), np.nan))
         for k in geo_ts_keys], index=df.index)
    df['d48_hr_curr'] = pd.Series([d48_hr_dict.get(k, np.nan) for k in geo_hr_keys], index=df.index)
    df['d48_hr_prev'] = pd.Series(
        [d48_hr_dict.get((k[0], k[1] - 1), np.nan) for k in geo_hr_keys], index=df.index)
    df['d49_same_hr'] = pd.Series([d49_hr_dict.get(k, np.nan) for k in geo_hr_keys], index=df.index)
    df['d49_prev_hr'] = pd.Series(
        [d49_hr_dict.get((k[0], k[1] - 1), np.nan) for k in geo_hr_keys], index=df.index)

    # Slot-smoothed d48 (±15 min rolling mean)
    df['d48_prev_slot']  = pd.Series([d48_prev_dict.get(k, np.nan)  for k in geo_ts_keys], index=df.index)
    df['d48_next_slot']  = pd.Series([d48_next_dict.get(k, np.nan)  for k in geo_ts_keys], index=df.index)
    df['d48_roll3_mean'] = pd.Series([d48_roll3_dict.get(k, np.nan) for k in geo_ts_keys], index=df.index)

    # d49 slot-level anchors (most recent known observations)
    df['d49_at_2_0']     = df['geohash'].map(d49_at_2_0_dict)
    df['d49_last2_mean'] = df['geohash'].map(d49_slot_last2_dict)

    # Ratio features — the core of the prediction strategy
    df['ratio_at_hour'] = df.apply(
        lambda r: get_ratio_at_hour(r['geohash'], r['geo4'], r['hour']), axis=1)
    df['ratio_at_h2']   = (df['geohash'].map(r2_dict)
                            .fillna(df['geohash'].map(geo_ratio_dict))
                            .fillna(1.5))
    df['ratio_slope']   = df['geohash'].map(ratio_slope_d).fillna(0)
    df['day_ratio']     = (df['geohash'].map(geo_ratio_dict)
                            .fillna(df['geo4'].map(geo4_ratio_dict))
                            .fillna(1.5))
    df['geo4_ratio']    = df['geo4'].map(geo4_ratio_dict).fillna(1.5)

    # Rule-based predictions (also used as post-processing anchor)
    df['d48_day_ratio_pred']    = df['d48_demand']    * df['day_ratio']
    df['d48_hour_ratio_pred']   = df['d48_demand']    * df['ratio_at_hour']
    df['d48_geo4_ratio_pred']   = df['d48_demand']    * df['geo4_ratio']
    df['d48_roll3_hratio_pred'] = df['d48_roll3_mean'] * df['ratio_at_hour']

    # Merge historical aggregates
    df = df.merge(geo_hour_d48, on=['geohash', 'hour'], how='left')
    df = df.merge(geo_d48,      on='geohash',           how='left')
    df = df.merge(d49_trend_df, on='geohash',           how='left')
    df = df.merge(d49_last_df,  on='geohash',           how='left')
    df = df.merge(geo4_hour,    on=['geo4', 'hour'],     how='left')
    df = df.merge(geo3_hour,    on=['geo3', 'hour'],     how='left')

    df['geo_hour_ratio_pred']  = df['geo_hour_mean'] * df['day_ratio']
    df['geo4_hour_ratio_pred'] = df['geo4_hour_mean'] * df['geo4_ratio']
    df['geo_hour_hratio_pred'] = df['geo_hour_mean'] * df['ratio_at_hour']

    df['d49_extrapolated'] = df.apply(
        lambda r: (
            d49_last_dict.get(r['geohash'], 0) +
            d49_trend_dict.get(r['geohash'], 0) *
            (r['hour'] - d49_lhour_dict.get(r['geohash'], 2))
        ), axis=1).clip(0, 1)

    df['d48_hr_vs_daily']    = df['d48_hr_curr'] / (df['geo_d48_mean'] + 1e-8)
    df['d48_day_slope']      = df['geohash'].map(d48_day_slope_dict).fillna(0)
    df['geo4_neighbor_mean'] = df.apply(
        lambda r: get_geo4_neighbor_mean(r['geohash'], r['geo4']), axis=1)

    return df


print('Building features...')
train_fe = make_features(train)
test_fe  = make_features(test)
print('Done')

In [8]:
# NaN fill and label encoding
gm = train['demand'].mean()

fill_cols = [
    'geo_hour_mean', 'geo_hour_std', 'geo_hour_med', 'geo_hour_q25', 'geo_hour_q75',
    'geo_d48_mean', 'geo_d48_std', 'geo_d48_med',
    'd49_trend', 'd49_last_demand', 'd49_last_hour',
    'd48_demand', 'd48_day_ratio_pred', 'd48_hour_ratio_pred', 'd48_geo4_ratio_pred',
    'd49_same_hr', 'geo4_hour_mean', 'geo3_hour_mean',
    'd49_extrapolated', 'day_ratio', 'geo4_ratio',
    'geo_hour_ratio_pred', 'geo4_hour_ratio_pred', 'geo_hour_hratio_pred',
    'd48_hr_prev', 'd48_hr_curr', 'd49_prev_hr', 'd48_hr_vs_daily',
    'ratio_at_hour', 'ratio_at_h2', 'ratio_slope',
    'd48_prev_slot', 'd48_next_slot', 'd48_roll3_mean', 'd48_roll3_hratio_pred',
    'd49_at_2_0', 'd49_last2_mean',
    'd48_day_slope', 'geo4_neighbor_mean',
]
for col in fill_cols:
    train_fe[col] = train_fe[col].fillna(gm)
    test_fe[col]  = test_fe[col].fillna(gm)

all_df = pd.concat([train_fe, test_fe])
for col in ['geohash', 'geo3', 'geo4', 'geo5']:
    le = LabelEncoder().fit(all_df[col])
    train_fe[col + '_enc'] = le.transform(train_fe[col])
    test_fe[col  + '_enc'] = le.transform(test_fe[col])

print('Encoding done')

## Model

In [9]:
FEATURES = [
    # Time
    'hour', 'minute', 'time_slot', 'time_of_day',
    'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
    'is_peak', 'is_night', 'day',
    # Location
    'lat', 'lon', 'geohash_enc', 'geo3_enc', 'geo4_enc', 'geo5_enc',
    # Road attributes
    'LV_enc', 'LM_enc', 'RT_enc', 'WX_enc', 'NumberofLanes', 'Temperature',
    # d48 cross-day core
    'd48_demand', 'd48_day_ratio_pred', 'd48_hour_ratio_pred', 'd48_geo4_ratio_pred',
    # Slot-smoothed d48
    'd48_prev_slot', 'd48_next_slot', 'd48_roll3_mean', 'd48_roll3_hratio_pred',
    # Historical geo × hour stats
    'geo_hour_mean', 'geo_hour_std', 'geo_hour_med', 'geo_hour_q25', 'geo_hour_q75',
    'geo_d48_mean', 'geo_d48_std', 'geo_d48_med', 'geo4_hour_mean', 'geo3_hour_mean',
    # Scaled predictions
    'geo_hour_ratio_pred', 'geo4_hour_ratio_pred', 'geo_hour_hratio_pred',
    # d49 early-morning signals
    'd49_same_hr', 'd49_trend', 'd49_last_demand', 'd49_extrapolated', 'd49_prev_hr',
    'd49_at_2_0', 'd49_last2_mean',
    # Ratio features
    'day_ratio', 'geo4_ratio', 'ratio_at_hour', 'ratio_at_h2', 'ratio_slope',
    # d48 lags
    'd48_hr_prev', 'd48_hr_curr', 'd48_hr_vs_daily',
    # Structural
    'd48_day_slope', 'geo4_neighbor_mean',
]

X      = train_fe[FEATURES]
y      = train_fe['demand']
X_test = test_fe[FEATURES]

# d49 rows carry 9× weight — the leaderboard only scores day 49
d49_mask       = (train['day'].values == 49)
sample_weights = np.where(d49_mask, 9.0, 1.0)

print(f'{len(FEATURES)} features | X={X.shape} | X_test={X_test.shape}')
print(f'd49 rows: {d49_mask.sum()} ({d49_mask.mean()*100:.1f}% of train)')

In [10]:
BASE_PARAMS = {
    'objective'         : 'regression',
    'metric'            : 'rmse',
    'learning_rate'     : 0.015,
    'num_leaves'        : 511,
    'min_child_samples' : 20,
    'feature_fraction'  : 0.8,
    'bagging_fraction'  : 0.8,
    'bagging_freq'      : 5,
    'reg_alpha'         : 0.05,
    'reg_lambda'        : 0.1,
    'n_estimators'      : 4000,
    'verbose'           : -1,
}

# Average over multiple seeds to cancel out LightGBM's bagging variance.
# Set SEEDS = [42] for a quick single-seed run.
SEEDS   = [42, 7, 123, 314, 999]
N_FOLDS = 5

all_oof   = np.zeros(len(X))
all_test  = np.zeros(len(X_test))
all_count = 0
seed_r2s  = []

print(f'{len(SEEDS)} seeds × {N_FOLDS} folds = {len(SEEDS) * N_FOLDS} models')
print('-' * 60)

for seed in SEEDS:
    params = {**BASE_PARAMS, 'random_state': seed}
    kf     = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof    = np.zeros(len(X))
    t_pred = np.zeros(len(X_test))

    for fold, (tri, vli) in enumerate(kf.split(X)):
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X.iloc[tri], y.iloc[tri],
            sample_weight=sample_weights[tri],
            eval_set=[(X.iloc[vli], y.iloc[vli])],
            callbacks=[
                lgb.early_stopping(150, verbose=False),
                lgb.log_evaluation(False)
            ]
        )
        oof[vli]  = m.predict(X.iloc[vli])
        t_pred   += m.predict(X_test) / N_FOLDS

    r2_d49 = r2_score(y.values[d49_mask], oof[d49_mask])
    r2_all = r2_score(y, oof)
    print(f'  seed {seed:>4}  r2_all={r2_all:.4f}  r2_d49={r2_d49:.4f}  score={r2_d49*100:.2f}')
    seed_r2s.append(r2_d49)

    all_oof   += oof
    all_test  += t_pred
    all_count += 1

all_oof  /= all_count
all_test /= all_count

r2_oof_all = r2_score(y, all_oof)
r2_oof_d49 = r2_score(y.values[d49_mask], all_oof[d49_mask])
print('-' * 60)
print(f'Ensemble  r2_all={r2_oof_all:.4f}  r2_d49={r2_oof_d49:.4f}  score={r2_oof_d49*100:.2f}')
print(f'Seed range: {min(seed_r2s):.4f} – {max(seed_r2s):.4f}  std={np.std(seed_r2s):.4f}')

## Feature importance

In [11]:
import matplotlib.pyplot as plt

params_imp = {**BASE_PARAMS, 'random_state': 42}
models_imp = []

for tri, vli in KFold(n_splits=5, shuffle=True, random_state=42).split(X):
    m = lgb.LGBMRegressor(**params_imp)
    m.fit(
        X.iloc[tri], y.iloc[tri],
        sample_weight=sample_weights[tri],
        eval_set=[(X.iloc[vli], y.iloc[vli])],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(False)]
    )
    models_imp.append(m)

avg_imp = np.mean([m.feature_importances_ for m in models_imp], axis=0)
fi_df   = (pd.DataFrame({'feature': FEATURES, 'importance': avg_imp})
             .sort_values('importance', ascending=False))

plt.figure(figsize=(10, 8))
plt.barh(fi_df['feature'][:25][::-1], fi_df['importance'][:25][::-1])
plt.xlabel('Importance')
plt.title('Top 25 features (avg over 5 folds)')
plt.tight_layout()
plt.show()

print(fi_df.head(20).to_string(index=False))

## Post-processing

A 5% blend toward the rule-based anchor (`d48 × ratio_at_hour`) softens any extreme predictions the model makes when extrapolating to high test hours. Only applied if it actually improves OOF d49.

In [12]:
BLEND_WEIGHT = 0.05

anchor_test  = test_fe['d48_hour_ratio_pred'].values.clip(0, 1)
anchor_train = train_fe['d48_hour_ratio_pred'].values.clip(0, 1)

test_preds_blend = (1 - BLEND_WEIGHT) * all_test + BLEND_WEIGHT * anchor_test
oof_blend        = (1 - BLEND_WEIGHT) * all_oof  + BLEND_WEIGHT * anchor_train

r2_blend_d49 = r2_score(y.values[d49_mask], oof_blend[d49_mask])
r2_raw_d49   = r2_score(y.values[d49_mask], all_oof[d49_mask])

print(f'OOF d49 raw    = {r2_raw_d49:.6f}')
print(f'OOF d49 blend  = {r2_blend_d49:.6f}')

if r2_blend_d49 >= r2_raw_d49:
    test_preds = test_preds_blend
    print(f'Using blended predictions (+{(r2_blend_d49 - r2_raw_d49)*100:.4f} pts)')
else:
    test_preds = all_test.copy()
    print('Blend did not help — using raw predictions')

test_preds = np.clip(test_preds, 0, 1)

## Submit

In [13]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission.to_csv('submission.csv', index=False)

print(f'Saved submission.csv  shape={submission.shape}')
print(submission['demand'].describe().round(4))
print(f'\nOOF d49 score: {max(r2_blend_d49, r2_raw_d49)*100:.4f}')